<a href="https://colab.research.google.com/github/ajeet3301/VoiceCraft/blob/main/VoiceCraft_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoiceCraft Studio & Telegram Bot (Free Google Colab T4 GPU)
Run your Hinglish Voice Cloning Studio and Telegram Bot with **Free NVIDIA T4 GPU acceleration**.

- **Speed**: ~1 second per sentence (10x faster than CPU)
- **Cost**: 100% Free via Google Colab + Cloudflare Tunnel
- **Features**: Web Studio, Telegram Bot, Multilingual TTS (v3), Hinglish language routing

---
### Quick Instructions:
1. In Google Colab top menu, click **Runtime** > **Change runtime type** > Select **T4 GPU** > Save.
2. Run **Step 1** to verify GPU.
3. Run **Step 2** to install dependencies.
4. Run **Step 3** to paste your Telegram Bot Token.
5. Run **Step 4** to launch and get your public web link!

In [1]:
# @title Step 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU.')


Wed Nov  4 04:11:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# @title Step 2: Clone Repository & Install Dependencies
import os
if not os.path.exists('VoiceCraft'):
    !git clone https://github.com/ajeet3301/VoiceCraft.git
%cd VoiceCraft

# Install Linux system packages
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 > /dev/null

# Install Python packages without overriding Colab CUDA PyTorch
!pip install -q -r requirements_colab.txt
!pip install -q -e . --no-deps

# Download portable Linux Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print('\nSetup completed successfully!')


Cloning into 'VoiceCraft'...
remote: Enumerating objects: 438, done.
remote: Counting objects: 100% (438/438), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 438 (delta 179), reused 415 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (438/438), 2.09 MiB | 26.81 MiB/s, done.
Resolving deltas: 100% (179/179), done.
/content/VoiceCraft
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... d

In [ ]:
# @title Step 3: Configure Telegram Bot & Model Settings
# @markdown Enter your Telegram bot credentials below (optional):
TELEGRAM_BOT_TOKEN = '9127866863:AAHzFWQVJW-qQnbkxYiN-kCpkhcyTgr11r8'  # @param {type:'string'}
MODEL_VERSION = 'v3'  # @param ['v3', 'v2', 'v1']
ADMIN_USER_ID = '8011886933'  # @param {type:'string'}

env_lines = [
    f'BOT_TOKEN={TELEGRAM_BOT_TOKEN.strip()}',
    f'TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN.strip()}',
    f'CHATTERBOX_MULTILINGUAL_T3_MODEL={MODEL_VERSION}',
]
if ADMIN_USER_ID.strip():
    env_lines.append(f'ADMIN_USER_ID={ADMIN_USER_ID.strip()}')

with open('.env', 'w', encoding='utf-8') as f:
    f.write('\n'.join(env_lines) + '\n')

print(f'Configuration saved. Model set to: {MODEL_VERSION}')
if TELEGRAM_BOT_TOKEN.strip() and TELEGRAM_BOT_TOKEN.strip() != 'your_telegram_bot_token_here':
    print('Telegram Bot is enabled and will start automatically with the server.')
else:
    print('No Telegram Bot token provided. Only Web Studio UI will be active.')


In [ ]:
# @title Step 4: Launch VoiceCraft Web Studio & Telegram Bot
import subprocess
import time
import re
from IPython.display import display, HTML

# Start Cloudflare Tunnel in background
print('Creating Cloudflare Tunnel for public access...')
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print('=' * 60)
    print(f'  YOUR APP IS LIVE AT: {public_url}')
    print('=' * 60)
    display(HTML(f'''
    <div style="background-color: #FAF9F5; border: 1px solid #C96442; border-radius: 8px; padding: 20px; margin: 15px 0; font-family: sans-serif;">
        <h3 style="color: #1F1E1D; margin-top: 0; font-family: Georgia, serif;">VoiceCraft Studio &amp; Bot Online</h3>
        <p style="color: #555; margin-bottom: 15px;">Open this public URL on your phone or PC browser:</p>
        <a href="{public_url}" target="_blank" style="background-color: #C96442; color: #FAF9F5; text-decoration: none; padding: 10px 22px; border-radius: 6px; font-weight: 600; display: inline-block;">
            Open VoiceCraft Studio &#8599;
        </a>
    </div>
    '''))
else:
    print('Could not automatically detect public link. Check tunnel output.')

# Launch server.py (auto-starts Telegram bot if token is configured)
print('\nStarting VoiceCraft server (FastAPI on port 7860)...')
!python server.py
